## Step 13 — expand zones_3 to zones_4 using voronoi analysis as guide
**# of cells in notebook:** 1

**Purpose:** In Juba, and I reckon this will be true in other cities, there is a problem of small low population --or 0 population-- 'islands' on the periphery. I believe this will be a problem in other cities because the workflow that creates city extents is very generous with the perihperal areas it includes. This introduces considerable complication when merging low population features, because merging is based on a polygon neighbors table where neighbors share edges. After experientation, I believe one of the cleanest ways to resolve the issue is to create voronoi polygons across zones_3 features and then to dissolve them by the `concat` value of the features within them. Within the dissolved voronois, we then dissolve zones_3 singlepart features into multipart features and sum their populations. This notebook is strictly concerned creating the voronois. To do so, we adapt a step (morphological tesselation) from the Urban Taxonomy package to `zones_3` features.     

**Input:**

- a geodatabase with `zones_3`, `zones_3_morph_tess`
  
**Output:** `zones_4` polygon layer

**Main logic:**

1. Dissolve `zones_3_morph_tess` by `concat`, using `SINGLE_PART`, to create `zones_3_morph_tess_DSLV`. This creates dissolved tessellation polygons that represent contiguous areas with the same `concat` value.
2. Convert `zones_3` polygons to inside points, then intersect those points with `zones_3_morph_tess_DSLV` so each original `zones_3` feature can be associated with the dissolved tessellation polygon that contains its inside point.
3. Spatially join the dissolved tessellation feature ID, `FID_zones_3_morph_tess_DSLV`, back to `zones_3`, creating `zones_3_SJ`. The script keeps all original `zones_3` fields and adds only that one tessellation ID field from the intersect result.
4. Dissolve `zones_3_SJ` by `FID_zones_3_morph_tess_DSLV` to create `zones_4`, using `MULTI_PART`. This groups together `zones_3` features that belong to the same dissolved tessellation area.
5. During the dissolve to `zones_4`, sum numeric fields that should accumulate, especially `population` and `block_count`, and preserve representative values for tracking fields using `FIRST`. The script then renames dissolved statistic fields by removing `SUM_` and `FIRST_` prefixes so the output field names are cleaner.

In [ ]:
import arcpy
import os

# ============================================================
# Integrated script through creation of zones_4
# Does NOT include zones_4a
# ============================================================

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False

zones_gdb = r"E:\World Bank deliverbale 1\_analysis\zones\zones.gdb"
voronoi_gdb = r"E:\World Bank deliverbale 1\_analysis\voronoi\voronoi.gdb"

# Inputs
zones_3 = os.path.join(zones_gdb, "zones_3")
zones_3_morph_tess = os.path.join(voronoi_gdb, "zones_3_morph_tess")

# Outputs
zones_3_morph_tess_DSLV = os.path.join(voronoi_gdb, "zones_3_morph_tess_DSLV")
zones_3_pnt = os.path.join(zones_gdb, "zones_3_pnt")
zones_3_pnt_itx = os.path.join(zones_gdb, "zones_3_pnt_itx")
zones_3_SJ = os.path.join(zones_gdb, "zones_3_SJ")
zones_4 = os.path.join(zones_gdb, "zones_4")


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def require_exists(path, label="Dataset"):
    if not arcpy.Exists(path):
        raise FileNotFoundError(f"{label} not found: {path}")


def require_field(fc, field_name):
    fields = [f.name for f in arcpy.ListFields(fc)]
    if field_name not in fields:
        raise ValueError(
            f"Field '{field_name}' not found in:\n{fc}\n\n"
            f"Available fields:\n" + "\n".join(fields)
        )


def delete_if_exists(path):
    if arcpy.Exists(path):
        arcpy.management.Delete(path)
        print(f"Deleted existing output: {path}")


def count_features(fc):
    return int(arcpy.management.GetCount(fc)[0])


# ============================================================
# STEP 1
# Dissolve zones_3_morph_tess on concat, singlepart output
# ============================================================

print("\n" + "=" * 70)
print("STEP 1: Creating zones_3_morph_tess_DSLV")
print("=" * 70)

require_exists(zones_3_morph_tess, "Input feature class")
require_field(zones_3_morph_tess, "concat")
delete_if_exists(zones_3_morph_tess_DSLV)

print("Dissolving zones_3_morph_tess on concat...")

arcpy.analysis.PairwiseDissolve(
    in_features=zones_3_morph_tess,
    out_feature_class=zones_3_morph_tess_DSLV,
    dissolve_field=["concat"],
    statistics_fields=None,
    multi_part="SINGLE_PART",
    concatenation_separator=""
)

print("Done.")
print(f"Input features:  {count_features(zones_3_morph_tess):,}")
print(f"Output features: {count_features(zones_3_morph_tess_DSLV):,}")
print(f"Output written to: {zones_3_morph_tess_DSLV}")


# ============================================================
# STEP 2
# Convert zones_3 polygons to inside points
# ============================================================

print("\n" + "=" * 70)
print("STEP 2: Creating zones_3_pnt")
print("=" * 70)

require_exists(zones_3, "Input feature class")
delete_if_exists(zones_3_pnt)

print("Creating inside points from zones_3...")

arcpy.management.FeatureToPoint(
    in_features=zones_3,
    out_feature_class=zones_3_pnt,
    point_location="INSIDE"
)

print("Done.")
print(f"Input polygon features: {count_features(zones_3):,}")
print(f"Output point features:  {count_features(zones_3_pnt):,}")
print(f"Output written to: {zones_3_pnt}")


# ============================================================
# STEP 3
# Intersect zones_3_pnt with zones_3_morph_tess_DSLV
# Output should be point
# ============================================================

print("\n" + "=" * 70)
print("STEP 3: Creating zones_3_pnt_itx")
print("=" * 70)

require_exists(zones_3_pnt, "Input point feature class")
require_exists(zones_3_morph_tess_DSLV, "Input polygon feature class")
delete_if_exists(zones_3_pnt_itx)

print("Intersecting zones_3_pnt with zones_3_morph_tess_DSLV...")

arcpy.analysis.Intersect(
    in_features=[
        zones_3_pnt,
        zones_3_morph_tess_DSLV
    ],
    out_feature_class=zones_3_pnt_itx,
    join_attributes="ALL",
    cluster_tolerance=None,
    output_type="POINT"
)

print("Done.")
print(f"zones_3_pnt features:              {count_features(zones_3_pnt):,}")
print(f"zones_3_morph_tess_DSLV features:  {count_features(zones_3_morph_tess_DSLV):,}")
print(f"Output intersect points:           {count_features(zones_3_pnt_itx):,}")
print(f"Output written to: {zones_3_pnt_itx}")


# ============================================================
# STEP 4
# Spatial join one field from zones_3_pnt_itx to zones_3
# Target: zones_3
# Join: zones_3_pnt_itx
# Keep all target features
# Match option: INTERSECT
# Keep all target fields and only FID_zones_3_morph_tess_DSLV from join
# ============================================================

print("\n" + "=" * 70)
print("STEP 4: Creating zones_3_SJ")
print("=" * 70)

target_fc = zones_3
join_fc = zones_3_pnt_itx
join_field_to_keep = "FID_zones_3_morph_tess_DSLV"

require_exists(target_fc, "Target feature class")
require_exists(join_fc, "Join feature class")
require_field(join_fc, join_field_to_keep)
delete_if_exists(zones_3_SJ)

print("Building field mappings...")

field_mappings = arcpy.FieldMappings()

# Keep all fields from zones_3
field_mappings.addTable(target_fc)

# Add only FID_zones_3_morph_tess_DSLV from zones_3_pnt_itx
fm = arcpy.FieldMap()
fm.addInputField(join_fc, join_field_to_keep)

out_field = fm.outputField
out_field.name = join_field_to_keep
out_field.aliasName = join_field_to_keep
fm.outputField = out_field

field_mappings.addFieldMap(fm)

print("Running spatial join...")

arcpy.analysis.SpatialJoin(
    target_features=target_fc,
    join_features=join_fc,
    out_feature_class=zones_3_SJ,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    field_mapping=field_mappings,
    match_option="INTERSECT"
)

out_fields = [f.name for f in arcpy.ListFields(zones_3_SJ)]

print("Done.")
print(f"Target zones_3 features:       {count_features(target_fc):,}")
print(f"Join zones_3_pnt_itx features: {count_features(join_fc):,}")
print(f"Output zones_3_SJ features:    {count_features(zones_3_SJ):,}")
print(f"Output written to: {zones_3_SJ}")

if join_field_to_keep in out_fields:
    print(f"Confirmed field added: {join_field_to_keep}")
else:
    print(f"WARNING: Expected field not found in output: {join_field_to_keep}")


# ============================================================
# STEP 5
# Dissolve zones_3_SJ on FID_zones_3_morph_tess_DSLV
# Multipart output
# Rename SUM_ and FIRST_ fields back to clean names
# ============================================================

print("\n" + "=" * 70)
print("STEP 5: Creating zones_4")
print("=" * 70)

input_fc = zones_3_SJ
output_fc = zones_4
dissolve_field = "FID_zones_3_morph_tess_DSLV"

require_exists(input_fc, "Input feature class")
require_field(input_fc, dissolve_field)
delete_if_exists(output_fc)

input_fields = [f.name for f in arcpy.ListFields(input_fc)]

requested_stats = [
    ["Join_Count", "SUM"],
    ["TARGET_FID", "FIRST"],
    ["concat", "FIRST"],
    ["zone2_id", "FIRST"],
    ["population", "SUM"],
    ["block_count", "SUM"],
    ["zone1", "FIRST"],
    ["admin", "FIRST"],
    ["density", "FIRST"],
    ["zone_mix", "FIRST"],
    ["merge_z1", "FIRST"],
    ["merge_admin", "FIRST"],
    ["merge_cluster", "FIRST"],
    ["merge_count", "FIRST"],
    ["used_reock", "FIRST"],
    ["reock_mrg_n", "FIRST"],
    ["FID_zones_3_morph_tess", "FIRST"],
]

statistics_fields = []
missing_stats_fields = []

for field_name, stat_type in requested_stats:
    if field_name in input_fields:
        statistics_fields.append([field_name, stat_type])
    else:
        missing_stats_fields.append(field_name)

if missing_stats_fields:
    print("WARNING: These requested statistics fields were not found and will be skipped:")
    for fld in missing_stats_fields:
        print(f"  - {fld}")

print(f"Using dissolve field: {dissolve_field}")
print("Dissolving zones_3_SJ to create zones_4...")

arcpy.analysis.PairwiseDissolve(
    in_features=input_fc,
    out_feature_class=output_fc,
    dissolve_field=[dissolve_field],
    statistics_fields=statistics_fields,
    multi_part="MULTI_PART",
    concatenation_separator=""
)

# ------------------------------------------------------------
# Rename dissolved statistic fields
# Remove FIRST_ and SUM_ prefixes
# ------------------------------------------------------------
print("Renaming dissolved statistic fields...")

existing_output_fields = [f.name for f in arcpy.ListFields(output_fc)]

rename_pairs = []

for field_name, stat_type in statistics_fields:
    dissolved_name = f"{stat_type}_{field_name}"
    clean_name = field_name

    if dissolved_name in existing_output_fields:
        rename_pairs.append((dissolved_name, clean_name))
    else:
        print(f"WARNING: Expected dissolved field not found: {dissolved_name}")

for old_name, new_name in rename_pairs:
    existing_output_fields = [f.name for f in arcpy.ListFields(output_fc)]

    if old_name not in existing_output_fields:
        print(f"Skipping rename; field not found: {old_name}")
        continue

    if new_name in existing_output_fields:
        print(
            f"Skipping rename {old_name} -> {new_name}; "
            f"target field name already exists."
        )
        continue

    print(f"Renaming {old_name} -> {new_name}")

    arcpy.management.AlterField(
        in_table=output_fc,
        field=old_name,
        new_field_name=new_name,
        new_field_alias=new_name
    )

print("Done.")
print(f"Input zones_3_SJ features: {count_features(input_fc):,}")
print(f"Output zones_4 features:   {count_features(output_fc):,}")
print(f"Output written to: {output_fc}")

print("\nStatistics used:")
for field_name, stat_type in statistics_fields:
    print(f"  {field_name}: {stat_type}")

print("\nFinal zones_4 fields:")
for f in arcpy.ListFields(output_fc):
    print(f"  {f.name}")

print("\nIntegrated script completed successfully through zones_4.")